# SMS Spam & Phishing Detection: full pipeline on Google Colab
Niraj Bankar (EN25552002) & Zaheer Mulani (EN25552018), MTech NLP mini-project.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Then run all cells.

The notebook clones the repository, downloads the public datasets, trains both models, evaluates them, exports the mobile ONNX model, rebuilds the report and slides from *your* results, and starts the Gradio demo with a public link. Results are saved to Google Drive so a disconnect does not lose them.

> GPU training is not bit-for-bit reproducible, so your numbers can differ slightly from the ones in the submitted report. Run the last cells to regenerate the report/slides from this run.

## 1. Get the code
Replace the URL with your GitHub repository (or upload the project ZIP to Colab and unzip it instead).

In [ ]:
REPO_URL = 'https://github.com/<your-username>/sms-spam-phishing-detection.git'
import os
if not os.path.exists('sms-spam-phishing-detection'):
    if os.path.exists('/content/SMS_Spam_Phishing_Detection_NLP_Project.zip'):
        !unzip -q /content/SMS_Spam_Phishing_Detection_NLP_Project.zip -d /content
    else:
        !git clone $REPO_URL
%cd /content/sms-spam-phishing-detection

In [ ]:
!pip -q install -r requirements.txt onnxruntime-extensions
import torch; print('GPU:', torch.cuda.is_available())

## 2. Optional: save outputs to Google Drive

In [ ]:
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/sms-phishing-results'; os.makedirs(DRIVE_DIR, exist_ok=True)

## 3. Data: download, mask personal data, label mapping, split

In [ ]:
!python -m src.data_loader | tail -40
!python -m src.eda

## 4. Classical baseline (TF-IDF + Logistic Regression, + SVM / NB)

In [ ]:
!python -m src.train_classical

## 5. Transformer: model survey and fine-tuning (about 5 minutes on a T4)

In [ ]:
!python -m src.model_selection
!python -m src.train_transformer --model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 --lr 8e-5
# optional comparison model:
# !python -m src.train_transformer --model distilbert-base-multilingual-cased --lr 3e-5

## 6. Mobile export: ONNX, INT8, vocabulary pruning, end-to-end model

In [ ]:
!python -m deployment.export_onnx
!python -m deployment.build_e2e_model

## 7. Evaluation on the held-out test set

In [ ]:
!python -m src.evaluate --reset --models logreg naive_bayes linear_svm transformer onnx_int8 onnx_pruned_int8
print(open('results/metrics/comparison_table.md').read())

In [ ]:
!python -m pytest -q tests

## 8. Rebuild report and presentation from this run's results

In [ ]:
!python docs/build_diagrams.py && python docs/build_report.py && python docs/build_presentation.py
if SAVE_TO_DRIVE:
    !cp -r results docs/*.docx docs/*.pptx models/optimized $DRIVE_DIR/

## 9. Demo (public link)
Opens a Gradio app. The `*.gradio.live` link works on a phone too; it expires after 72 hours or when the Colab session stops.

In [ ]:
!python app/app.py --share